# ⚡ Electricity Supply Forecasting - Out-of-Sample Future Validation (Jan – Mar 2026)

This notebook validates the final **Tuned LSTM Model** (`Supply_LSTM.keras`) against real out-of-sample observations for **January, February, and March 2026**.

### Workflow Steps:
1. **Load Trained Model & Scalers**: Load `Supply_LSTM.keras`, `Supply_LSTM_X_scaler.pkl`, and `Supply_LSTM_y_scaler.pkl`.
2. **Load Dataset**: Read `supply_dataset.csv` containing actual observations.
3. **Construct 3D LSTM Input Sequences**: Extract 6-month historical sliding windows ending at target months.
4. **Predict & Inverse Scale**: Predict scaled supply and convert back to original **MU** unit using `target_scaler.inverse_transform()`.
5. **Evaluate & Display**: Compare predicted supply vs actual supply and calculate MAE, RMSE, MAPE, and R².

In [1]:
# ============================================================
# STEP 1: LOAD TRAINED TUNED LSTM MODEL & SCALERS
# ============================================================

import joblib
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_FILE = 'Supply_LSTM.keras'
X_SCALER_FILE = 'Supply_LSTM_X_scaler.pkl'
Y_SCALER_FILE = 'Supply_LSTM_y_scaler.pkl'

# Load trained LSTM model and scalers
model = load_model(MODEL_FILE)
feature_scaler = joblib.load(X_SCALER_FILE)
target_scaler = joblib.load(Y_SCALER_FILE)

print('========================================')
print('MODEL & SCALERS LOADED SUCCESSFULLY')
print('========================================')
print('Model File          :', MODEL_FILE)
print('Feature Scaler File :', X_SCALER_FILE)
print('Target Scaler File  :', Y_SCALER_FILE)

MODEL & SCALERS LOADED SUCCESSFULLY
Model File          : Supply_LSTM.keras
Feature Scaler File : Supply_LSTM_X_scaler.pkl
Target Scaler File  : Supply_LSTM_y_scaler.pkl


In [2]:
# ============================================================
# STEP 2: LOAD & PREPARE SUPPLY DATASET
# ============================================================

df = pd.read_csv('supply_dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print('Dataset Shape :', df.shape)
print('Start Date    :', df['Date'].min().strftime('%Y-%m-%d'))
print('End Date      :', df['Date'].max().strftime('%Y-%m-%d'))

print('\nTarget Months (Jan - Mar 2026):')
target_rows = df[df['Date'].isin(pd.to_datetime(['2026-01-01', '2026-02-01', '2026-03-01']))]
print(target_rows[['Date', 'Total']])

Dataset Shape : (120, 29)
Start Date    : 2016-04-01
End Date      : 2026-03-01

Target Months (Jan - Mar 2026):
          Date     Total
117 2026-01-01  10189.56
118 2026-02-01  10405.60
119 2026-03-01  11247.51


In [3]:
# ============================================================
# STEP 3: DEFINE 18 MODEL FEATURES & LOOKBACK SEQUENCE LENGTH
# ============================================================

FEATURES = [
    'Coal_Lag_1', 'Oil_Gas_Lag_1', 'Nuclear_Lag_1', 'Hydro_Lag_1',
    'Solar_Lag_1', 'Wind_Lag_1', 'Small_Hydro_Lag_1', 'Bio_Power_Lag_1',
    'Year', 'Month', 'Quarter', 'Total_Lag_1', 'Total_Lag_3',
    'Total_Lag_6', 'Total_Lag_12', 'Total_Rolling_3', 'Total_Rolling_6', 'Total_Rolling_12'
]

LOOKBACK = 6  # 6-month sequence length selected during tuning

print('Feature Count:', len(FEATURES))
print('Lookback     :', LOOKBACK, 'months')
print('\nFeatures List:')
for f in FEATURES:
    print(' -', f)

Feature Count: 18
Lookback     : 6 months

Features List:
 - Coal_Lag_1
 - Oil_Gas_Lag_1
 - Nuclear_Lag_1
 - Hydro_Lag_1
 - Solar_Lag_1
 - Wind_Lag_1
 - Small_Hydro_Lag_1
 - Bio_Power_Lag_1
 - Year
 - Month
 - Quarter
 - Total_Lag_1
 - Total_Lag_3
 - Total_Lag_6
 - Total_Lag_12
 - Total_Rolling_3
 - Total_Rolling_6
 - Total_Rolling_12


In [4]:
# ============================================================
# STEP 4: GENERATE SEQUENTIAL FUTURE PREDICTIONS (JAN - MAR 2026)
# ============================================================

target_dates = ['2026-01-01', '2026-02-01', '2026-03-01']
predictions_mu = []
scaled_predictions = []

for target_date in target_dates:
    # Get index for current target month
    idx = df[df['Date'] == target_date].index[0]
    
    # Extract 6-month sequence ending at target_date
    seq_df = df.iloc[idx - LOOKBACK + 1 : idx + 1]
    
    # Raw feature matrix (shape: 6 timesteps, 18 features)
    X_raw = seq_df[FEATURES].values
    
    # Scale using pre-fitted training feature scaler
    X_scaled = feature_scaler.transform(X_raw)
    
    # Reshape into 3D tensor for LSTM: (1 sample, 6 timesteps, 18 features)
    X_input = X_scaled.reshape(1, LOOKBACK, len(FEATURES))
    
    # Predict scaled target value
    pred_scaled = model.predict(X_input, verbose=0)
    scaled_predictions.append(pred_scaled[0, 0])
    
    # Inverse transform to original MU unit
    pred_mu = target_scaler.inverse_transform(pred_scaled)[0, 0]
    predictions_mu.append(pred_mu)

# Ground truth actual target values for comparison
actual_rows = df[df['Date'].isin(pd.to_datetime(target_dates))].copy()
actuals_mu = actual_rows['Total'].values

print('Forecasting complete for Jan, Feb, and Mar 2026.')

Forecasting complete for Jan, Feb, and Mar 2026.


In [5]:
# ============================================================
# STEP 5: DISPLAY FORECAST VALIDATION RESULTS & METRICS
# ============================================================

month_names = ['January 2026', 'February 2026', 'March 2026']

print('========================================')
print('SUPPLY FORECAST VALIDATION')
print('========================================\n')

for i in range(3):
    m_name = month_names[i]
    p_val = predictions_mu[i]
    a_val = actuals_mu[i]
    err = abs(a_val - p_val)
    print(f'{m_name}')
    print(f'Predicted Supply : {p_val:.2f} MU')
    print(f'Actual Supply    : {a_val:.2f} MU')
    print(f'Error            : {err:.2f} MU\n')

# Calculate evaluation metrics across Jan-Mar 2026
mae = mean_absolute_error(actuals_mu, predictions_mu)
rmse = np.sqrt(mean_squared_error(actuals_mu, predictions_mu))
mape = np.mean(np.abs((actuals_mu - predictions_mu) / actuals_mu)) * 100
r2 = r2_score(actuals_mu, predictions_mu)

print('========================================')
print('PERFORMANCE METRICS (JAN - MAR 2026)')
print('========================================')
print(f'MAE  : {mae:.2f} MU')
print(f'RMSE : {rmse:.2f} MU')
print(f'MAPE : {mape:.2f} %')
print(f'R²   : {r2:.4f}')

SUPPLY FORECAST VALIDATION

January 2026
Predicted Supply : 8809.63 MU
Actual Supply    : 10189.56 MU
Error            : 1379.93 MU

February 2026
Predicted Supply : 9229.95 MU
Actual Supply    : 10405.60 MU
Error            : 1175.65 MU

March 2026
Predicted Supply : 9574.64 MU
Actual Supply    : 11247.51 MU
Error            : 1672.87 MU

PERFORMANCE METRICS (JAN - MAR 2026)
MAE  : 1409.48 MU
RMSE : 1424.18 MU
MAPE : 13.24 %
R²   : -8.7371
